In [0]:
# Imports
from datetime import datetime, timezone
from pathlib import Path
import json
import unicodedata
from typing import Any

import pandas as pd

# Caminhos
BRONZE_ROOT = Path("/Volumes/workspace/conectatel/raw_files/bronze")
SILVER_ROOT = Path("/Volumes/workspace/conectatel/raw_files/silver")
SILVER_ROOT.mkdir(parents=True, exist_ok=True)

# Colunas
EXPECTED_COLUMNS = [
    "chamado_id", "data_abertura", "canal", "categoria",
    "subcategoria", "estado", "cidade", "duracao_minutos",
    "resolvido_primeiro_contato", "encaminhado_humano",
    "satisfacao_1_a_5", "plano_atual", "resumo_atendimento",
]
TEXT_COLUMNS = [
    "canal", "categoria", "subcategoria",
    "estado", "cidade", "plano_atual",
]

def normalize_text(value: Any) -> str:
    """Normaliza texto e remove acentos."""
    text = "unknown" if pd.isna(value) else str(value)
    normalized = unicodedata.normalize("NFKD", text)
    return "".join(
        char for char in normalized
        if not unicodedata.combining(char)
    ).strip().lower()

def validate_columns(frame: pd.DataFrame) -> None:
    """Valida as colunas obrigatórias do dataset."""
    missing = [
        column for column in EXPECTED_COLUMNS
        if column not in frame.columns
    ]
    if missing:
        raise ValueError(f"Colunas ausentes: {missing}")

def generate_quality_report(raw, clean, before, after) -> dict[str, Any]:
    """Gera qualidade antes, depois e regras de negócio."""
    raw_dates = pd.to_datetime(raw["data_abertura"], errors="coerce")
    raw_duration = pd.to_numeric(raw["duracao_minutos"], errors="coerce")
    raw_satisfaction = pd.to_numeric(
        raw["satisfacao_1_a_5"], errors="coerce"
    )
    today = pd.Timestamp.now().normalize()
    tokens = {"true", "false", "1", "0", "sim", "nao", "não", "yes", "no"}
    invalid_booleans = {}
    for column in ["resolvido_primeiro_contato", "encaminhado_humano"]:
        values = raw[column].dropna().astype("string")
        values = values.str.strip().str.lower()
        invalid_booleans[column] = int((~values.isin(tokens)).sum())

    business_checks = {
        "missing_id_count": int(raw["chamado_id"].isna().sum()),
        "future_date_count": int(raw_dates.gt(today).fillna(False).sum()),
        "duration_negative_count": int(
            raw_duration.lt(0).fillna(False).sum()
        ),
        "satisfaction_out_of_range_count": int(
            (raw_satisfaction.lt(1) | raw_satisfaction.gt(5))
            .fillna(False).sum()
        ),
        "boolean_invalid_count": invalid_booleans,
    }
    return {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "before": {
            "rows": int(len(raw)),
            "exact_duplicates": int(raw.duplicated(keep=False).sum()),
            "missing_values": {
                column: int(raw[column].isna().sum())
                for column in raw.columns
            },
            "date_invalid_count": int(
                (raw["data_abertura"].notna() & raw_dates.isna()).sum()
            ),
            "duration_non_numeric_count": int(
                (raw["duracao_minutos"].notna() & raw_duration.isna()).sum()
            ),
            "duration_negative_count": business_checks[
                "duration_negative_count"
            ],
            "satisfaction_out_of_range_count": business_checks[
                "satisfaction_out_of_range_count"
            ],
        },
        "after": {
            "rows": int(after),
            "removed_duplicate_rows": int(before - after),
            "missing_values": {
                column: int(clean[column].isna().sum())
                for column in clean.columns
            },
        },
        "business_checks": business_checks,
    }

def generate_schema_report(
    frame: pd.DataFrame,
    output_path: Path,
) -> dict[str, Any]:
    """Gera o schema do DataFrame limpo."""
    return {
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "path": str(output_path),
        "columns": [
            {
                "name": str(column),
                "pandas_dtype": str(frame[column].dtype),
            }
            for column in frame.columns
        ],
    }

In [0]:
# Leitura
SNAPSHOT_PATH = BRONZE_ROOT / "bronze_calls_snapshot.csv"
if not SNAPSHOT_PATH.exists():
    raise FileNotFoundError(f"Snapshot não encontrado: {SNAPSHOT_PATH}")

calls = pd.read_csv(
    SNAPSHOT_PATH,
    dtype="string",
    keep_default_na=True,
    na_values=["", "NA", "NaN", "NULL", "null"],
)
validate_columns(calls)
unexpected_columns = [
    column for column in calls.columns
    if column not in EXPECTED_COLUMNS
]
print(f"Linhas brutas: {len(calls)}")
print(f"Colunas extras: {unexpected_columns}")

In [0]:
# Limpeza
pd.set_option("future.no_silent_downcasting", True)

cleaned = calls.copy()
for column in TEXT_COLUMNS:
    cleaned[column] = cleaned[column].map(normalize_text)

cleaned["data_abertura"] = pd.to_datetime(
    cleaned["data_abertura"], errors="coerce"
).fillna(pd.Timestamp("1900-01-01"))

for column in ["duracao_minutos", "satisfacao_1_a_5"]:
    cleaned[column] = pd.to_numeric(cleaned[column], errors="coerce")
    fallback = 0 if cleaned[column].dropna().empty else cleaned[column].median()
    cleaned[column] = cleaned[column].fillna(fallback)

boolean_map = {
    "true": True, "1": True, "sim": True, "yes": True,
    "false": False, "0": False, "nao": False, "não": False, "no": False,
}
for column in ["resolvido_primeiro_contato", "encaminhado_humano"]:
    values = (
        cleaned[column].fillna("unknown").astype("string")
        .str.strip().str.lower().map(boolean_map)
    )
    cleaned[column] = values.astype("boolean").fillna(False).astype(bool)

cleaned["satisfacao_1_a_5"] = cleaned["satisfacao_1_a_5"].clip(1, 5)
cleaned["resumo_atendimento"] = (
    cleaned["resumo_atendimento"].fillna("unknown")
    .astype("string").str.strip()
)
before_rows = len(cleaned)
cleaned = cleaned.drop_duplicates(keep="first").copy()
after_rows = len(cleaned)
CLEANED_PATH = SILVER_ROOT / "silver_calls_cleaned.csv"
cleaned.to_csv(CLEANED_PATH, index=False)
print(f"Linhas antes: {before_rows}")
print(f"Linhas depois: {after_rows}")
print(f"Duplicatas removidas: {before_rows - after_rows}")

In [0]:
# Relatório
quality_report = generate_quality_report(
    raw=calls,
    clean=cleaned,
    before=before_rows,
    after=after_rows,
)
(SILVER_ROOT / "silver_quality_report.json").write_text(
    json.dumps(quality_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

schema_report = generate_schema_report(
    frame=cleaned,
    output_path=CLEANED_PATH,
)
(SILVER_ROOT / "silver_schema.json").write_text(
    json.dumps(schema_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Silver: {after_rows} linhas")

In [0]:
# Metadados
metadata_source = BRONZE_ROOT / "bronze_corpus_metadata.json"
metadata_target = SILVER_ROOT / "silver_corpus_metadata.json"
if not metadata_source.exists():
    raise FileNotFoundError(
        f"Metadados não encontrados: {metadata_source}"
    )
metadata_target.write_text(
    metadata_source.read_text(encoding="utf-8"),
    encoding="utf-8",
)
print(f"Metadados copiados: {metadata_target}")